# 07 - PAvPU Evaluation
Computes Patch Accuracy vs Patch Uncertainty (PAvPU) across all the models

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.utils import set_global_seed, load_config, get_logger
from data.paths import get_dataset_config, get_split_file, get_checkpoints_dir, get_results_dir
from src.classes import get_class_registry
from src.features import DEFAULT_FEATURE_NAMES
from src.pavpu import compute_pavpu
from src.metrics import compute_metrics, results_to_dict

logger = get_logger("07_evaluate_pavpu")

In [ ]:
DATASET = "ahn3"

config = load_config("default")
set_global_seed(config["experiment"]["seed"])
dataset_cfg = get_dataset_config(DATASET)
class_registry = get_class_registry(DATASET)
num_classes = class_registry.num_classes
feature_columns = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES

In [ ]:
test_df = pd.read_csv(get_split_file(DATASET, "test"))

feature_columns   = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES
feature_columns_aug = feature_columns + ["__u_norm__"]

X_test     = test_df[feature_columns].values.astype(np.float32)
X_test_aug = test_df[feature_columns_aug].values.astype(np.float32)
y_test     = test_df["class_idx"].values.astype(np.int32)
u_test     = test_df["__u_norm__"].values.astype(np.float32)
xyz_test   = test_df[["x", "y", "z"]].values.astype(np.float32)

k = config["pavpu"]["k_neighbors"]
u_threshold = config["pavpu"].get("uncertainty_threshold", None)
predict_batch_size = config["training"].get("batch_size", 512)

print(f"Test: {len(test_df):,} points | k={k} | u_threshold={u_threshold or 'median (per-run)'}")

## Load all three trained models

Each model is loaded from its checkpoint saved in notebooks 04-06.

In [ ]:
models = {}
for name, ckpt_name in [
    ("baseline",            "baseline"),
    ("uncertainty_feature", "uncertainty_feature"),
    ("uncertainty_loss",    "uncertainty_loss"),
]:
    ckpt_path = get_checkpoints_dir(DATASET, ckpt_name) / "best.keras"
    if ckpt_path.exists():
        models[name] = tf.keras.models.load_model(str(ckpt_path))
        print(f"Loaded {name} from {ckpt_path}")
    else:
        print(f"Checkpoint not found for '{name}': {ckpt_path}")
        print(f"Run notebook 04/05/06 first.")

## Compute metrics and PAvPU for each model

All three models are evaluated on **exactly the same** test rows, with **the same** uncertainty values and **the same** KNN patch definition - so PAvPU scores are directly comparable.

In [ ]:
predictions = {}
for model_name, model in models.items():
    X_input = X_test_aug if model_name == "uncertainty_feature" else X_test
    predictions[model_name] = np.argmax(
        model.predict(X_input, batch_size=predict_batch_size, verbose=0), axis=1
    )

all_results = []
metrics_by_model = {}

for model_name, y_pred in predictions.items():
    metrics = compute_metrics(
        y_test, y_pred,
        num_classes=class_registry.num_classes,
        class_names=class_registry.class_names_in_order(),
        excluded_classes=dataset_cfg.get("classes_excluded_from_metrics", []),
    )
    metrics_by_model[model_name] = metrics

    pavpu_score, pavpu_details = compute_pavpu(
        y_true=y_test,
        y_pred=y_pred,
        uncertainty=u_test,
        xyz=xyz_test,
        k=k,
        uncertainty_threshold=u_threshold,
    )

    row = {
        "model": model_name,
        "accuracy": metrics["accuracy"],
        "mIoU": metrics["mIoU"],
        "PAvPU": pavpu_score,
        "N_AC": pavpu_details["N_AC"],
        "N_AU": pavpu_details["N_AU"],
        "N_IC": pavpu_details["N_IC"],
        "N_IU": pavpu_details["N_IU"],
        "u_threshold": pavpu_details["uncertainty_threshold"],
    }
    all_results.append(row)

    res_dir = get_results_dir(DATASET, model_name)
    pd.DataFrame([pavpu_details]).to_csv(res_dir / "pavpu.csv", index=False)

results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

## Comparison table - accuracy, mIoU, PAvPU

In [ ]:
display_cols = ["model", "accuracy", "mIoU", "PAvPU"]
print(results_df[display_cols].to_string(index=False))

results_df.to_csv(
    get_results_dir(DATASET, "comparison") / "all_models.csv", index=False
)
print("\nSaved to results/comparison/all_models.csv")

## Visualisation: AC/AU/IC/IU breakdown per model

A good uncertainty-aware model should have:
- High **AC** (accurate and certain - model knows what it knows)
- High **IU** (inaccurate and uncertain - model knows what it doesn't know)
- Low **IC** (inaccurate and certain - dangerous: confident wrong predictions)
- Low **AU** (accurate and uncertain - wasted uncertainty signal)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df["model"], results_df["PAvPU"], color=["steelblue","tomato","seagreen"])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("PAvPU")
axes[0].set_title("PAvPU across models")
for i, v in enumerate(results_df["PAvPU"]):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

categories = ["N_AC", "N_AU", "N_IC", "N_IU"]
colors     = ["#2ecc71", "#f1c40f", "#e74c3c", "#3498db"]
labels     = ["AC (accurate & certain)", "AU (accurate & uncertain)",
              "IC (inaccurate & certain)", "IU (inaccurate & uncertain)"]

bottoms = np.zeros(len(results_df))
for cat, color, label in zip(categories, colors, labels):
    vals = results_df[cat].values.astype(float)
    totals = results_df[["N_AC","N_AU","N_IC","N_IU"]].sum(axis=1).values
    pcts = vals / totals
    axes[1].bar(results_df["model"], pcts, bottom=bottoms, color=color, label=label)
    bottoms += pcts

axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Fraction of patches")
axes[1].set_title("Patch category breakdown per model")
axes[1].legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(
    str(get_results_dir(DATASET, "comparison") / "pavpu_comparison.png"),
    dpi=120, bbox_inches="tight",
)
plt.show()

## Per-class IoU heatmap
Visual comparison of per-class IoU across all three models - shows whether uncertainty-aware training helps especially on minority classes

In [ ]:
# Reuses metrics_by_model computed in the cell above - no re-predicting.
iou_data = {
    model_name: m["per_class_iou"].set_index("class_name")["IoU"]
    for model_name, m in metrics_by_model.items()
}
iou_df = pd.DataFrame(iou_data)
print(iou_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, max(4, len(iou_df) * 0.5)))
im = ax.imshow(iou_df.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(iou_df.columns))); ax.set_xticklabels(iou_df.columns, rotation=15, ha="right")
ax.set_yticks(range(len(iou_df.index)));  ax.set_yticklabels(iou_df.index, fontsize=9)
plt.colorbar(im, ax=ax, label="IoU")
ax.set_title("Per-class IoU - model comparison")
for i in range(len(iou_df.index)):
    for j in range(len(iou_df.columns)):
        ax.text(j, i, f"{iou_df.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(
    str(get_results_dir(DATASET, "comparison") / "per_class_iou_heatmap.png"),
    dpi=120, bbox_inches="tight",
)
plt.show()